<a href="https://colab.research.google.com/github/SonalGire/Langchain/blob/main/Currency_conversion_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 LangChain Tool Calling – Currency Conversion :

This project demonstrates how to **create and use custom tools in LangChain** and allow an **LLM (Large Language Model) to automatically call and execute those tools**.

The example implements a **Currency Conversion Assistant 💱** that can fetch real-time exchange rates and convert currencies using external APIs.

The goal of this project is to help developers understand the **complete workflow of LangChain tools**, including tool creation, tool binding, tool calling, and tool execution.

---

## 🚀 What This Project Does

This assistant can:

🔹 Get **real-time currency exchange rates** using an external API
🔹 Convert a currency amount from **one currency to another**
🔹 Allow the **LLM to decide which tool to use automatically**
🔹 Execute the tool and return the final result to the user

Example:

User Question
👉 *"Convert 100 USD to INR"*

System Flow
👉 LLM → Fetch Exchange Rate → Convert Currency → Return Final Result

---

## 🧠 Concepts Covered

This repository helps you understand the **core LangChain tool workflow**:

### 1️⃣ Tool Creation

Create custom tools using LangChain's `@tool` decorator.

Example:

* Get currency conversion factor
* Convert currency amount

### 2️⃣ Tool Binding

Bind the tools to the LLM so the model knows which tools it can use.

### 3️⃣ Tool Calling

The LLM analyzes the user query and **decides which tool should be called**.

### 4️⃣ Tool Execution

The selected tool is executed with the required parameters and the result is sent back to the model.

---

## 🔄 Project Workflow

User Query
⬇
LLM understands the request
⬇
LLM selects the appropriate tool
⬇
Tool executes the function
⬇
Result returned to LLM
⬇
Final response shown to the user

---

## 🛠 Technologies Used

🐍 Python
🔗 LangChain
🤖 OpenAI / Azure OpenAI
🌐 Exchange Rate API

---

## 🎯 Purpose of This Project

This project is designed for developers who want to learn:

✔ How **LLMs interact with external tools**
✔ How **LangChain tool calling works internally**
✔ How to build **AI assistants that can perform real tasks using APIs**

It serves as a **simple educational example for understanding LangChain Tool Calling architecture**.

---

⭐ If you find this project useful, feel free to **star the repository** and explore the code!


In [ ]:
API_KEY = "6be4355c9d6aec473456vkhs"

In [ ]:
!pip install langchain-openai langchain-core langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


In [67]:
from langchain_openai import ChatOpenAI, AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [68]:
# Tool creation

@tool
def get_conversion_factor(base_currency: str, target_currency:str) -> float:
  """This function fetches the currency conversion factor b/w a given base currency and a target currency."""
  url = f"https://v6.exchangerate-api.com/v6/6be435a2d6aec47/pair/{base_currency}/{target_currency}"

  response = requests.get(url)
  return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """Given a currency conversion rate this function calculates the target currency value from a given  base currency value"""
  return base_currency_value * conversion_rate


In [69]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773273601,
 'time_last_update_utc': 'Thu, 12 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1773360001,
 'time_next_update_utc': 'Fri, 13 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.2032}

In [70]:
convert.invoke({'base_currency_value':10, 'conversion_rate':92.203})

922.03

In [71]:
AZURE_OPENAI_API_KEY = "57Q7tODX8IQkwNIifqSOWBjFXJ3w3AAABACOG7R80"
AZURE_OPENAI_ENDPOINT = "https://east-us-instance.ai.zk.com/"


llm = AzureChatOpenAI(
    azure_deployment="ykt-4",
    api_version="2024-12-01-preview",
    temperature=0.7,
    api_key=AE_AI_API_KEY,
    azure_endpoint=RE_AI_ENDPOINT
)


In [72]:
# tool binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [73]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr?')]

In [74]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr?', additional_kwargs={}, response_metadata={})]

In [75]:
ai_message = llm_with_tools.invoke(messages)


In [58]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_KcXjyf07fdEEn1jaqAtLWo6d',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_W5GtWQBPo9pxnDu7ONwZS8kj',
  'type': 'tool_call'}]

In [76]:
messages.append(ai_message)

In [77]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    #  fetch the conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message in message list
    messages.append(tool_message1)

  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current argument
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [78]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 122, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DISSdiWH9BoK5iJxd4bQ6bDmH2836', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False,

In [80]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is **92.2032**. Based on this rate, 10 USD converts to **922.03 INR**.'